# MiniMax H3 — ComfyUI + Cloudflare Preflight First

This is the main H3 notebook. The important change is the order:

1. Install ComfyUI + required custom nodes only.
2. Start ComfyUI and the named Cloudflare Tunnel.
3. Open `https://comfy.zetbros.com` and verify the full ComfyUI UI works.
4. **STOP here by default. No H3 model weights download yet.**
5. Only after you confirm the UI works, enable the model-download gate and continue.
6. Download the H3 model stack.
7. Restart only ComfyUI once; Cloudflare stays connected.

This avoids spending Colab units on large model downloads before the public ComfyUI connection is proven.

**Important:** keep only one Colab runtime connected to this Cloudflare tunnel. Disconnect any old tunnel-test runtime first.

## 0. Mount Drive + persistence

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PERSIST_MODELS_TO_DRIVE = False   # False = faster model loading from Colab VM disk
PERSIST_OUTPUT_TO_DRIVE = True    # Keep ComfyUI output/user data across restarts
DRIVE_ROOT = '/content/drive/MyDrive/MiniMax_H3_ComfyUI'


## 1. Check GPU + safe runtime settings

In [ ]:
import subprocess, os

def sh(cmd):
    return subprocess.check_output(cmd, shell=True, text=True).strip()

gpu_name = sh('nvidia-smi --query-gpu=name --format=csv,noheader | head -n1')
vram_mb = int(sh('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | head -n1'))
vram_gb = vram_mb / 1024
name = gpu_name.lower()

if 'rtx pro 6000' in name or 'blackwell' in name:
    H3_RESERVE_VRAM_GB = '6'
elif 'a100' in name:
    H3_RESERVE_VRAM_GB = '4'
elif 'l4' in name:
    H3_RESERVE_VRAM_GB = '2'
else:
    H3_RESERVE_VRAM_GB = '1'

print(f'GPU: {gpu_name} ({vram_gb:.1f} GB)')
print('Reserved VRAM:', H3_RESERVE_VRAM_GB, 'GB')


## 2. Clone/update the H3 branch

In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy


## 3. Install/update ComfyUI + H3 custom nodes — NO model weights yet

This installs ComfyUI and the custom-node code needed by your H3/Director workflows. It does **not** download the large H3 model weights.

Installer output appears live below and is appended to `/content/h3_comfy_logs/setup.log`. A failure reports its stage and shell line. Installation must succeed before preflight or any model download.

In [ ]:
import os, subprocess, sys

os.environ['COMFY_ROOT'] = '/content/ComfyUI'
os.environ['H3_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['H3_PERSIST_MODELS'] = '1' if PERSIST_MODELS_TO_DRIVE else '0'
os.environ['H3_PERSIST_OUTPUT'] = '1' if PERSIST_OUTPUT_TO_DRIVE else '0'
os.environ['H3_VRAM_MODE'] = 'auto'
os.environ['H3_RESERVE_VRAM_GB'] = H3_RESERVE_VRAM_GB
os.environ['H3_PREVIEW_METHOD'] = 'none'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

sys.path.insert(0, '/content/minimax_h3_comfy')
import importlib, comfy_preflight
importlib.reload(comfy_preflight)  # Pick up fixes after rerunning the branch-fetch cell.
run_setup_command = comfy_preflight.run_setup_command

# Stream both stdout and stderr into this cell; append a redacted setup log.
run_setup_command(['bash', '/content/minimax_h3_comfy/install_comfy_h3.sh'],
                  label='ComfyUI + H3 installation')

CUSTOM='/content/ComfyUI/custom_nodes'
def clone_or_pull(url, folder):
    path=f'{CUSTOM}/{folder}'
    if os.path.isdir(path+'/.git'):
        run_setup_command(['git','-C',path,'pull','--ff-only'], label=folder + ' update')
    else:
        run_setup_command(['git','clone','--depth','1',url,path], label=folder + ' clone')
    req=os.path.join(path,'requirements.txt')
    if os.path.exists(req):
        run_setup_command([sys.executable,'-m','pip','install','-r',req], label=folder + ' requirements')

clone_or_pull('https://github.com/AIMixer/ComfyUI_MiniMaxH3_Director.git','ComfyUI_MiniMaxH3_Director')
clone_or_pull('https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git','ComfyUI-VideoHelperSuite')
clone_or_pull('https://github.com/kijai/ComfyUI-KJNodes.git','ComfyUI-KJNodes')
clone_or_pull('https://github.com/pixaroma/ComfyUI-Pixaroma.git','ComfyUI-Pixaroma')
run_setup_command([sys.executable,'-m','pip','install','-U','huggingface_hub'], label='Hugging Face Hub package')

print('✅ ComfyUI + H3/Director custom nodes installed.')
print('✅ No H3 model weights have been downloaded by this notebook yet.')


## 4. Load Cloudflare tunnel token

Cloudflare Published Application must point to:

`comfy.zetbros.com` → `http://127.0.0.1:8188`

The Colab secret must be named `CF_TUNNEL_TOKEN`. A raw token or the full Cloudflare install command is accepted. The command is never executed. The preflight installs cloudflared if needed after the real ComfyUI origin is healthy.

In [ ]:
import sys
sys.path.insert(0, '/content/minimax_h3_comfy')
from comfy_preflight import read_token

# userdata.get runs in the notebook; pass the extracted token privately to the launcher.
_cf_token = read_token(use_environment=False)
print('Cloudflare tunnel token loaded from Colab Secrets.')


## 5. PRE-FLIGHT — Start ComfyUI + Cloudflare BEFORE model downloads

This is the checkpoint you asked for. When this cell succeeds, open `https://comfy.zetbros.com`. You should see the full ComfyUI interface, although model dropdowns will be empty/missing H3 weights until you continue later.

In [ ]:
import sys, importlib

H3_PREFLIGHT_COMPLETE = False
H3_MODELS_DOWNLOADED = False
PROCEED_WITH_H3_MODEL_DOWNLOADS = False
sys.path.insert(0, '/content/minimax_h3_comfy')
import comfy_preflight
importlib.reload(comfy_preflight)
comfy_preflight.run_launcher('preflight', token=_cf_token)
H3_PREFLIGHT_COMPLETE = True
print('Browser verification is still required; no model download is approved yet.')


## 6. HARD STOP / approval gate

The default is `False`, so **Run all stops here before downloading any H3 weights**.

First open `https://comfy.zetbros.com`. If the full ComfyUI UI works, change the value below to `True` and run this cell again, then continue to Section 7.

In [ ]:
PROCEED_WITH_H3_MODEL_DOWNLOADS = False

if PROCEED_WITH_H3_MODEL_DOWNLOADS is not True:
    print('Verify https://comfy.zetbros.com first.\n'
          'If the full ComfyUI UI loads, set\n'
          'PROCEED_WITH_H3_MODEL_DOWNLOADS = True\n'
          'and continue.')
    raise SystemExit('Paused before model downloads. ComfyUI + Cloudflare remain running.')

if globals().get('H3_PREFLIGHT_COMPLETE') is not True:
    raise SystemExit('Run the ComfyUI + Cloudflare preflight cell first.')
print('Approved. Continue to Section 7.')


## 7. Download the H3 model stack — only after approval

In [ ]:
# Guard this cell too: running it directly must never bypass approval.
if (globals().get('PROCEED_WITH_H3_MODEL_DOWNLOADS') is not True
        or globals().get('H3_PREFLIGHT_COMPLETE') is not True):
    raise SystemExit('Verify https://comfy.zetbros.com first, then approve in Section 6.')
H3_MODELS_DOWNLOADED = False
import subprocess
comfy_preflight.run_launcher('check')

from huggingface_hub import hf_hub_download
from pathlib import Path
import os, shutil

MODEL_ROOT = Path(f'{DRIVE_ROOT}/models' if PERSIST_MODELS_TO_DRIVE else '/content/ComfyUI/models')
for folder in ['diffusion_models','text_encoders','vae','loras','latent_upscale_models']:
    (MODEL_ROOT/folder).mkdir(parents=True, exist_ok=True)

if PERSIST_MODELS_TO_DRIVE:
    local_latent = Path('/content/ComfyUI/models/latent_upscale_models')
    drive_latent = MODEL_ROOT/'latent_upscale_models'
    drive_latent.mkdir(parents=True, exist_ok=True)
    if local_latent.is_symlink():
        local_latent.unlink()
    elif local_latent.exists():
        shutil.rmtree(local_latent) if local_latent.is_dir() else local_latent.unlink()
    local_latent.symlink_to(drive_latent, target_is_directory=True)

downloads = [
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_fl2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors', MODEL_ROOT, MODEL_ROOT/'diffusion_models'/'minimax_h3_ref2va_pruned_int8_convrot.safetensors'),
    ('Comfy-Org/MiniMax-H3','text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', MODEL_ROOT, MODEL_ROOT/'text_encoders'/'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_video_vae_fp16.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_video_vae_fp16.safetensors'),
    ('Comfy-Org/MiniMax-H3','vae/minimax_h3_audio_vae_fp32.safetensors', MODEL_ROOT, MODEL_ROOT/'vae'/'minimax_h3_audio_vae_fp32.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_ref2v_turbo_8step_v1.0_768p_comfyui_bf16.safetensors'),
    ('lightx2v/Minimax-h3-Turbo','minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors', MODEL_ROOT/'loras', MODEL_ROOT/'loras'/'minimax_h3_fl2v_turbo_8step_v1.0_comfyui_bf16.safetensors'),
    ('LBH-123-AI/Minimax_h3_latent_Upscaler','minimax_h3_latent_upscaler_3d_fp16.safetensors', MODEL_ROOT/'latent_upscale_models', MODEL_ROOT/'latent_upscale_models'/'minimax_h3_latent_upscaler_3d_fp16.safetensors')
]

for repo_id, filename, local_dir, target in downloads:
    if target.exists() and target.stat().st_size > 1024*1024:
        print('SKIP', target.name)
        continue
    print('DOWNLOAD', filename)
    hf_hub_download(repo_id=repo_id, filename=filename, local_dir=str(local_dir))
    if not target.exists():
        raise FileNotFoundError(f'Expected model was not created: {target}')
    print('READY', target.name)

print('✅ H3 model stack ready at:', MODEL_ROOT)

H3_MODELS_DOWNLOADED = True


## 8. Restart only ComfyUI so the new models appear

The Cloudflare connector is left running. The public URL stays the same.

In [ ]:
if (globals().get('PROCEED_WITH_H3_MODEL_DOWNLOADS') is not True
        or globals().get('H3_MODELS_DOWNLOADED') is not True):
    raise SystemExit('Complete the approved model-download cell before the final restart.')
import subprocess
comfy_preflight.run_launcher('restart')


## Diagnostics — safe before model downloads

Run this cell at any time after Section 2, including after the intentional stop. It reports the local HTTP status, listener, connector PIDs and the last 50 lines of each log, with tokens/credentials redacted. It does not start services or download models.

Local health and tunnel registration do not prove browser UI/Access/WebSocket success. In your browser, confirm the real ComfyUI canvas and menus load and the connection stays active before approving. After generation, download/save your outputs before disconnecting and deleting the Colab runtime; you control both steps.

In [ ]:
import sys, importlib
sys.path.insert(0, '/content/minimax_h3_comfy')
import comfy_preflight
importlib.reload(comfy_preflight)
comfy_preflight.diagnostics()
